# Four-carrier sender calibration, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

Uses only `data/plan4_dev.jsonl` (30 families, 10 per level, 90 variants). Do **not** load `plan_dev`, `plan_fresh`, `data/test.jsonl`, `hop_test`, or a `plan_test` file.

Do **not** resume Study 0 or Study 1 folders.

This is sender calibration, not the six-condition communication comparison and not compression.

- Level 1: capacity, refrigeration, and driver given directly.
- Level 2: A computes effective capacity from vehicle capacity, productive hours, and loading rate.
- Level 3: A first computes productive hours after travel and breaks, then effective capacity.
- A modes: no extra rollout (`a_direct`), written calculations (`a_written`), 40 latent steps then the same compact result lines (`a_latent40`).
- Base and B-change reuse the same saved A states.
- B receives **solver** intermediates only (`b_oracle`). Around 90% is a diagnostic target, not a validity proof.

If latent steps do not help, keep that result. Do not redesign the task until they win.

Download `/kaggle/working/results_stage_plan4_cal` during the run.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_cal.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_cal.py").is_file() and (cand / "data" / "plan4_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_cal.py and data/plan4_dev.jsonl. Re-upload the repo zip.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_cal.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan4_dev", sum(1 for _ in (REPO / "data" / "plan4_dev.jsonl").open(encoding="utf-8")))
assert not (REPO / "data" / "plan_test.jsonl").exists()

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_cal")
print("Do not resume Study 1. No compressor. No six-condition comparison.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_cal", "--split", "plan4_dev"])
print("plan4 cal exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_cal"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_cal" / "run"
print("reading", src)
for name in ("summary.json", "manifest.json"):
    path = src / name
    if path.is_file():
        print("====", name)
        print(path.read_text(encoding="utf-8")[:12000])